# A1 — Catalog Enrichment / doc2query (Gemini)

One-time, resumable, cached enrichment of the 47k-track catalog: for each track, Gemini writes short retrieval queries/blurb that get appended to its doc (closes the conversational↔metadata vocabulary gap — the top recall lever per P0). Output: an enriched parquet on Drive consumed by `Catalog.id_to_metadata(enriched=True)`. Spec: `30_A1_catalog_assets.md`.

## 1. Drive + Gemini key (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'; os.environ['HF_HOME']=f'{DRIVE}/hf_cache'
OUT=f'{DRIVE}/outputs'; os.makedirs(OUT,exist_ok=True); os.makedirs(os.environ['HF_HOME'],exist_ok=True)
GEMINI_KEY=userdata.get('GEMINI_API_KEY')
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=t
    from huggingface_hub import login; login(t)
except Exception as e: print('no HF_TOKEN:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s pandas numpy google-generativeai
import sys; sys.path.insert(0,'.')

## 3. Config

In [ ]:
MODEL='gemini-2.5-flash-lite'   # 2.0-flash-lite shut down 2026-06-01
N_REQUESTS=4
LIMIT=0          # *** SMOKE TEST knob *** set to e.g. 200 to enrich only 200 tracks (~1 min, cents); 0 = full 47k
CHECKPOINT=2000; WORKERS=16
OUT_PARQUET=f'{OUT}/catalog_enriched.parquet'; ORG='talkpl-ai'

## 4. Load catalog + few-shot style anchors (real train utterances) + Gemini generate_fn

In [ ]:
import google.generativeai as genai, random
from datasets import load_dataset
from mcrs.enrich.doc2query import build_enrich_prompt, enriched_document
from mcrs.data.ids import canonical_track_id
rows = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')

# Few-shot STYLE anchors: real listener queries from the TRAIN split (never dev/blind),
# stratified by conversation_goal.category so the synthetic queries cover the real styles.
train = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='train')
rng = random.Random(42); by_cat = {}
for s in train.select(range(min(2000, len(train)))):
    g = (s.get('conversation_goal') or {}).get('category') or 'other'
    utts = [e['content'] for e in s['conversations'] if e['role']=='user']
    if utts: by_cat.setdefault(g, []).append(utts[0])
EXAMPLES = []
for g, utts in by_cat.items(): EXAMPLES += rng.sample(utts, min(2, len(utts)))
EXAMPLES = EXAMPLES[:10]
print(f'{len(EXAMPLES)} style exemplars across {len(by_cat)} goal categories:')
for e in EXAMPLES: print('  -', e[:80])

genai.configure(api_key=GEMINI_KEY)
SYSTEM, _ = build_enrich_prompt(rows[0], n_requests=N_REQUESTS, examples=EXAMPLES)
gmodel = genai.GenerativeModel(MODEL, system_instruction=SYSTEM,
          generation_config={'max_output_tokens':256, 'temperature':0.7})
def gen(meta):
    _, user = build_enrich_prompt(meta, n_requests=N_REQUESTS, examples=EXAMPLES)
    try: return gmodel.generate_content(user).text
    except Exception: return ''

## 5. Resumable enrich loop (checkpoint to Drive)

In [ ]:
import os, pandas as pd
from concurrent.futures import ThreadPoolExecutor
done = {}
if os.path.exists(OUT_PARQUET):
    prev = pd.read_parquet(OUT_PARQUET); done = dict(zip(prev['track_id'], prev['enriched_doc']))
    print('resuming:', len(done), 'already enriched')
all_rows = list(rows)[:LIMIT] if LIMIT else list(rows)
pending = [r for r in all_rows if canonical_track_id(r['track_id']) not in done]
print(len(pending),'pending of',len(all_rows))
def _one(r):
    return canonical_track_id(r['track_id']), enriched_document(r, gen(r))
def _flush():
    pd.DataFrame({'track_id':list(done), 'enriched_doc':list(done.values())}).to_parquet(OUT_PARQUET, index=False)
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    for i,(tid,doc) in enumerate(ex.map(_one, pending),1):
        done[tid]=doc
        if i % CHECKPOINT == 0: _flush(); print('  checkpoint',i)
_flush(); print('DONE ->', OUT_PARQUET, '| total', len(done))

## 6. Validate: raw vs enriched BM25 recall@100 (100 dev sessions, CPU)

In [ ]:
from mcrs.data.catalog import Catalog
from mcrs.data.conversations import Conversations
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.eval.probe import recall_ceiling
enr = dict(zip(pd.read_parquet(OUT_PARQUET)['track_id'], pd.read_parquet(OUT_PARQUET)['enriched_doc']))
cat_raw = Catalog(rows); cat_enr = Catalog(rows, enriched_docs=enr)
conv = Conversations(load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='test').select(range(100)))
turns=list(conv.turns()); qs=[QueryBuilder().build(t).text for t in turns]
golds=[conv.gold(t.session_id,t.turn_number) for t in turns]
raw = BM25Channel(cat_raw, enriched=False).batch_text_to_item_retrieval(qs, 200)
enrl= BM25Channel(cat_enr, enriched=True ).batch_text_to_item_retrieval(qs, 200)
r_raw=recall_ceiling({'bm25':raw}, golds, ks=[20,100,200])['per_channel']['bm25']['recall']
r_enr=recall_ceiling({'bm25':enrl},golds, ks=[20,100,200])['per_channel']['bm25']['recall']
print('BM25 recall@100  raw=',round(r_raw[100],3),' enriched=',round(r_enr[100],3),
      ' (gate: enriched should lift recall; else drop A1)')